In [ ]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu


# The dataset is not included in this repository.
# Download it separately from DataShop and place it inside a local data folder.
DATA_PATH = Path("data/dataset.txt")

OUTPUT_DIR = Path("outputs")
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


df = pd.read_csv(
    DATA_PATH,
    sep="\t",
    low_memory=False
)

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)

In [ ]:
# INITIAL DATA INSPECTION

print("Data structure:")
df.info()

print("\nProblem distribution:")
print(
    df["Problem Name"]
    .value_counts(dropna=False)
)

print("\nStudent Response Type distribution:")
print(
    df["Student Response Type"]
    .value_counts(dropna=False)
)

print("\nTop 20 most frequent actions:")
print(
    df["Action"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nTop feedback classifications:")
print(
    df["Feedback Classification"]
    .value_counts(dropna=False)
    .head(10)
)

In [ ]:
# PREPROCESS KEY COLUMNS

# Parse timestamps
df["Time"] = pd.to_datetime(
    df["Time"],
    errors="coerce"
)

# Convert event-level duration to numeric values
df["Duration (sec)"] = pd.to_numeric(
    df["Duration (sec)"],
    errors="coerce"
)

print("Minimum timestamp:", df["Time"].min())
print("Maximum timestamp:", df["Time"].max())

print(
    "Missing/unparseable timestamps:",
    df["Time"].isna().sum()
)

print(
    "Missing/non-numeric duration values:",
    df["Duration (sec)"].isna().sum()
)

In [ ]:
# IDENTIFY HINT-RELATED ACTIVITY

hint_events = df[
    df["Student Response Type"] == "HINT_REQUEST"
]

print("Problems containing hint requests:")
print(
    hint_events["Problem Name"]
    .value_counts(dropna=False)
)

print("\nHint feedback classifications:")
print(
    hint_events["Feedback Classification"]
    .value_counts(dropna=False)
)

In [ ]:
# FILTER TO THE ANALYTICAL TASK: guess1Lab

guess1 = df[
    df["Problem Name"] == "guess1Lab"
].copy()

print("guess1Lab shape:", guess1.shape)

print(
    "Unique students:",
    guess1["Anon Student Id"].nunique()
)

print(
    "Unique sessions:",
    guess1["Session Id"].nunique()
)

print(
    "Total hint requests:",
    (
        guess1["Student Response Type"]
        == "HINT_REQUEST"
    ).sum()
)

print("\nStudent Response Type distribution:")
print(
    guess1["Student Response Type"]
    .value_counts(dropna=False)
)

print("\nTop 20 actions in guess1Lab:")
print(
    guess1["Action"]
    .value_counts(dropna=False)
    .head(20)
)

print("\nHint feedback classifications in guess1Lab:")
print(
    guess1.loc[
        guess1["Student Response Type"] == "HINT_REQUEST",
        "Feedback Classification"
    ].value_counts(dropna=False)
)

In [ ]:
# BUILD SESSION-LEVEL BEHAVIOURAL FEATURES

session_summary = (
    guess1
    .groupby("Session Id")
    .agg(
        anon_student_id=(
            "Anon Student Id",
            "first"
        ),

        n_events=(
            "Session Id",
            "size"
        ),

        total_duration_sec=(
            "Duration (sec)",
            "sum"
        ),

        n_hint_requests=(
            "Student Response Type",
            lambda x: (x == "HINT_REQUEST").sum()
        ),

        n_run_clicks=(
            "Action",
            lambda x: (x == "Block.clickRun").sum()
        ),

        n_block_grabbed=(
            "Action",
            lambda x: (x == "Block.grabbed").sum()
        ),

        n_block_snapped=(
            "Action",
            lambda x: (x == "Block.snapped").sum()
        ),

        n_input_edits=(
            "Action",
            lambda x: (x == "InputSlot.edited").sum()
        ),

        n_errors=(
            "Action",
            lambda x: (x == "Error").sum()
        ),

        n_category_changes=(
            "Action",
            lambda x: (x == "IDE.changeCategory").sum()
        )
    )
    .reset_index()
)


# Binary session-level hint indicator
session_summary["has_hint"] = (
    session_summary["n_hint_requests"] > 0
)

print(
    "Session-level dataset shape:",
    session_summary.shape
)

session_summary.head()

In [ ]:
# VALIDATE SESSION AND LEARNER COUNTS

print(
    "Number of sessions:",
    session_summary["Session Id"].nunique()
)

print(
    "Number of unique learners:",
    session_summary["anon_student_id"].nunique()
)

sessions_per_student = (
    session_summary
    .groupby("anon_student_id")["Session Id"]
    .nunique()
)

print("\nSessions per learner:")
print(
    sessions_per_student
    .value_counts()
    .sort_index()
)

print("\nLearners contributing more than one session:")
print(
    sessions_per_student[
        sessions_per_student > 1
    ]
)

In [ ]:
# SESSION GROUP COUNTS

session_counts = (
    session_summary["has_hint"]
    .value_counts()
    .sort_index()
)

print("Session counts by hint use:")
print(session_counts)

print(
    "\nNon-hint sessions:",
    (~session_summary["has_hint"]).sum()
)

print(
    "Hint sessions:",
    session_summary["has_hint"].sum()
)

In [ ]:
# BEHAVIOURAL METRICS USED IN THE ANALYSIS

metrics = [
    "n_events",
    "total_duration_sec",
    "n_run_clicks",
    "n_block_grabbed",
    "n_block_snapped",
    "n_input_edits",
    "n_errors",
    "n_category_changes"
]

metric_labels = {
    "n_events":
        "Number of events",

    "total_duration_sec":
        "Aggregated recorded duration (sec)",

    "n_run_clicks":
        "Run actions",

    "n_block_grabbed":
        "Block-grab actions",

    "n_block_snapped":
        "Block-snap actions",

    "n_input_edits":
        "Input edits",

    "n_errors":
        "Recorded error events",

    "n_category_changes":
        "Category changes"
}

In [ ]:
# OVERALL SESSION-LEVEL DESCRIPTIVE STATISTICS

overall_descriptive = (
    session_summary[metrics]
    .describe()
    .T
)

overall_descriptive

In [ ]:
# MEAN SESSION METRICS BY HINT USE

comparison_raw = (
    session_summary
    .groupby("has_hint")[metrics]
    .mean()
)

print("Unrounded group means:")
comparison_raw

In [ ]:
# FORMAT VALUES FOR REPORTING

def round_half_up(value, digits=2):
    quantizer = Decimal(
        "1." + ("0" * digits)
    )

    return float(
        Decimal(str(value)).quantize(
            quantizer,
            rounding=ROUND_HALF_UP
        )
    )


comparison_display = (
    comparison_raw
    .apply(
        lambda column:
        column.map(
            lambda value:
            round_half_up(value, 2)
        )
    )
)

comparison_display.index = [
    "Non-hint",
    "Hint"
]

comparison_display

In [ ]:
# GROUP MEDIANS FOR THE MAIN VISUAL COMPARISONS

median_comparison = (
    session_summary
    .groupby("has_hint")[
        [
            "n_events",
            "total_duration_sec"
        ]
    ]
    .median()
)

median_comparison.index = [
    "Non-hint",
    "Hint"
]

median_comparison

In [ ]:
# FIGURE 5.1
# MEAN SESSION-LEVEL BEHAVIOURAL METRICS

NON_HINT_COLOR = "#3274A1"
HINT_COLOR = "#E1812C"

non_hint_n = (
    ~session_summary["has_hint"]
).sum()

hint_n = (
    session_summary["has_hint"]
).sum()

fig, axes = plt.subplots(
    4,
    2,
    figsize=(14, 10)
)

axes = axes.flatten()

for ax, metric in zip(
    axes,
    metrics
):

    values = [
        comparison_raw.loc[
            False,
            metric
        ],

        comparison_raw.loc[
            True,
            metric
        ]
    ]

    bars = ax.bar(
        ["Non-hint", "Hint"],
        values,
        color=[
            NON_HINT_COLOR,
            HINT_COLOR
        ]
    )

    ax.set_title(
        metric_labels[metric],
        loc="left",
        fontweight="bold"
    )

    ax.grid(
        axis="y",
        alpha=0.3
    )

    ax.set_axisbelow(True)

    upper = max(values)

    if upper == 0:
        ax.set_ylim(0, 1)
    else:
        ax.set_ylim(
            0,
            upper * 1.35
        )

    for bar, value in zip(
        bars,
        values
    ):
        ax.text(
            bar.get_x()
            + bar.get_width() / 2,

            bar.get_height()
            + (
                upper * 0.04
                if upper > 0
                else 0.02
            ),

            f"{round_half_up(value, 2):.2f}",

            ha="center",
            va="bottom",
            fontweight="bold"
        )

    ax.set_xticks([])


fig.suptitle(
    "Mean session-level behavioural metrics",
    fontsize=18,
    fontweight="bold",
    y=0.995
)

fig.text(
    0.5,
    0.965,
    "guess1Lab sessions grouped by hint use",
    ha="center",
    fontsize=11
)


from matplotlib.patches import Patch

legend_handles = [
    Patch(
        color=NON_HINT_COLOR,
        label=f"Non-hint (n={non_hint_n})"
    ),

    Patch(
        color=HINT_COLOR,
        label=f"Hint (n={hint_n})"
    )
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=2,
    frameon=False
)

fig.text(
    0.5,
    0.01,
    (
        "Values are arithmetic means calculated across "
        "sessions; panels use separate y-axis scales."
    ),
    ha="center",
    fontsize=9
)

plt.tight_layout(
    rect=[0, 0.04, 1, 0.90]
)

figure_path = (
    FIGURE_DIR
    / "mean_session_metrics.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", figure_path)

In [ ]:
# FIGURE 5.2
# NUMBER OF EVENTS BY HINT USE

non_hint_events = (
    session_summary.loc[
        session_summary["has_hint"] == False,
        "n_events"
    ]
    .dropna()
)

hint_events_session = (
    session_summary.loc[
        session_summary["has_hint"] == True,
        "n_events"
    ]
    .dropna()
)

event_groups = [
    non_hint_events,
    hint_events_session
]

fig, ax = plt.subplots(
    figsize=(10, 6.5)
)

bp = ax.boxplot(
    event_groups,

    tick_labels=[
        f"Non-hint (n={len(non_hint_events)})",
        f"Hint (n={len(hint_events_session)})"
    ],

    widths=0.4,
    patch_artist=True,
    showfliers=False,

    medianprops={
        "color": "black",
        "linewidth": 2
    }
)

bp["boxes"][0].set_facecolor(
    NON_HINT_COLOR
)

bp["boxes"][1].set_facecolor(
    HINT_COLOR
)

for box in bp["boxes"]:
    box.set_alpha(0.35)


# Reproducible horizontal jitter
rng = np.random.default_rng(42)

for i, (values, colour) in enumerate(
    zip(
        event_groups,
        [
            NON_HINT_COLOR,
            HINT_COLOR
        ]
    ),
    start=1
):

    jitter = rng.normal(
        i,
        0.035,
        size=len(values)
    )

    ax.scatter(
        jitter,
        values,
        alpha=0.45,
        s=28,
        color=colour
    )


# Median annotations
for i, values in enumerate(
    event_groups,
    start=1
):

    median = values.median()

    ax.text(
        i + 0.23,
        median,
        f"Median {median:.1f}",
        va="center",
        fontweight="bold"
    )


ax.set_title(
    "Number of events by hint usage",
    fontsize=17,
    fontweight="bold",
    pad=30
)

ax.text(
    0.5,
    1.02,
    "guess1Lab session-level distribution",
    transform=ax.transAxes,
    ha="center",
    fontsize=11
)

ax.set_ylabel(
    "Number of events",
    fontweight="bold"
)

ax.grid(
    axis="y",
    alpha=0.3
)

ax.set_axisbelow(True)

fig.text(
    0.5,
    0.015,
    (
        "Boxes show the interquartile range; whiskers "
        "extend to 1.5 × IQR; points represent "
        "individual sessions."
    ),
    ha="center",
    fontsize=9
)

plt.tight_layout(
    rect=[0, 0.05, 1, 1]
)

figure_path = (
    FIGURE_DIR
    / "events_boxplot.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", figure_path)

In [ ]:
# FIGURE 5.3
# AGGREGATED RECORDED DURATION BY HINT USE

non_hint_duration = (
    session_summary.loc[
        session_summary["has_hint"] == False,
        "total_duration_sec"
    ]
    .dropna()
)

hint_duration = (
    session_summary.loc[
        session_summary["has_hint"] == True,
        "total_duration_sec"
    ]
    .dropna()
)

duration_groups = [
    non_hint_duration,
    hint_duration
]

fig, ax = plt.subplots(
    figsize=(10, 6.5)
)

bp = ax.boxplot(
    duration_groups,

    tick_labels=[
        f"Non-hint (n={len(non_hint_duration)})",
        f"Hint (n={len(hint_duration)})"
    ],

    widths=0.4,
    patch_artist=True,
    showfliers=False,

    medianprops={
        "color": "black",
        "linewidth": 2
    }
)

bp["boxes"][0].set_facecolor(
    NON_HINT_COLOR
)

bp["boxes"][1].set_facecolor(
    HINT_COLOR
)

for box in bp["boxes"]:
    box.set_alpha(0.35)


rng = np.random.default_rng(42)

for i, (values, colour) in enumerate(
    zip(
        duration_groups,
        [
            NON_HINT_COLOR,
            HINT_COLOR
        ]
    ),
    start=1
):

    jitter = rng.normal(
        i,
        0.035,
        size=len(values)
    )

    ax.scatter(
        jitter,
        values,
        alpha=0.45,
        s=28,
        color=colour
    )


for i, values in enumerate(
    duration_groups,
    start=1
):

    median = values.median()

    ax.text(
        i + 0.23,
        median,
        f"Median {median:.2f}",
        va="center",
        fontweight="bold"
    )


ax.set_title(
    "Aggregated recorded duration by hint usage",
    fontsize=17,
    fontweight="bold",
    pad=30
)

ax.text(
    0.5,
    1.02,
    "guess1Lab session-level distribution",
    transform=ax.transAxes,
    ha="center",
    fontsize=11
)

ax.set_ylabel(
    "Aggregated recorded duration (sec)",
    fontweight="bold"
)

ax.grid(
    axis="y",
    alpha=0.3
)

ax.set_axisbelow(True)

fig.text(
    0.5,
    0.015,
    (
        "Boxes show the interquartile range; whiskers "
        "extend to 1.5 × IQR; points represent "
        "individual sessions."
    ),
    ha="center",
    fontsize=9
)

plt.tight_layout(
    rect=[0, 0.05, 1, 1]
)

figure_path = (
    FIGURE_DIR
    / "duration_boxplot.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", figure_path)

In [ ]:
# MANN-WHITNEY U TESTS
#
# Convention:
# The hint group is supplied as the first argument.
# Therefore, U_statistic is the U statistic corresponding
# to the hint-session sample.

with_hint = session_summary[
    session_summary["has_hint"] == True
]

without_hint = session_summary[
    session_summary["has_hint"] == False
]

results = []

for metric in metrics:

    stat, p = mannwhitneyu(
        with_hint[metric],
        without_hint[metric],
        alternative="two-sided"
    )

    results.append({
        "metric":
            metric,

        "with_hint_mean":
            with_hint[metric].mean(),

        "without_hint_mean":
            without_hint[metric].mean(),

        "U_statistic":
            stat,

        "p_value":
            p
    })


test_results = pd.DataFrame(
    results
)

test_results["difference"] = (
    test_results["with_hint_mean"]
    -
    test_results["without_hint_mean"]
)

test_results

In [ ]:
# FORMAT STATISTICAL RESULTS FOR REPORTING

test_results_display = (
    test_results[
        [
            "metric",
            "U_statistic",
            "p_value"
        ]
    ]
    .copy()
)

test_results_display[
    "p_value"
] = (
    test_results_display[
        "p_value"
    ]
    .round(4)
)

test_results_display

In [ ]:
# OPTIONAL EXPLORATORY TRACE OF ONE HINT SESSION

hint_session_ids = (
    guess1.loc[
        guess1["Student Response Type"]
        == "HINT_REQUEST",
        "Session Id"
    ]
    .dropna()
    .unique()
)

print(
    "Number of sessions containing hints:",
    len(hint_session_ids)
)

if len(hint_session_ids) > 0:

    example_session_id = (
        hint_session_ids[0]
    )

    hint_sample = (
        guess1[
            guess1["Session Id"]
            == example_session_id
        ]
        .sort_values("Time")
    )

    display(
        hint_sample[
            [
                "Time",
                "Student Response Type",
                "Step Name",
                "Selection",
                "Action",
                "Feedback Classification",
                "Duration (sec)"
            ]
        ].head(80)
    )

In [ ]:
# FINAL REPRODUCIBILITY CHECKS

assert df.shape == (172160, 34)

assert guess1.shape[0] == 28435

assert (
    guess1["Anon Student Id"]
    .nunique()
    == 65
)

assert (
    guess1["Session Id"]
    .nunique()
    == 66
)

assert (
    session_summary["has_hint"]
    .sum()
    == 32
)

assert (
    (~session_summary["has_hint"])
    .sum()
    == 34
)

assert (
    session_summary["n_hint_requests"]
    .sum()
    == 309
)

assert np.isclose(
    session_summary.loc[
        ~session_summary["has_hint"],
        "n_events"
    ].mean(),
    393.97058823529414
)

assert np.isclose(
    session_summary.loc[
        session_summary["has_hint"],
        "n_events"
    ].mean(),
    470.0
)

assert np.isclose(
    session_summary.loc[
        ~session_summary["has_hint"],
        "total_duration_sec"
    ].median(),
    2248.4995
)

assert np.isclose(
    session_summary.loc[
        session_summary["has_hint"],
        "total_duration_sec"
    ].median(),
    2559.499
)

print(
    "All validation checks passed successfully."
)

In [ ]:
# SAVE FINAL ANALYTICAL TABLES

session_summary.to_csv(
    OUTPUT_DIR / "session_summary.csv",
    index=False
)

comparison_raw.to_csv(
    OUTPUT_DIR / "mean_session_metrics.csv"
)

test_results.to_csv(
    OUTPUT_DIR / "mann_whitney_results.csv",
    index=False
)

print(
    "Saved session-level dataset and result tables to:"
)

print(OUTPUT_DIR)

print("\nSaved dissertation figures to:")
print(FIGURE_DIR)